In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [3]:
from Backend_FastAPI.TodoApp.database import SessionLocal
from Backend_FastAPI.TodoApp.udemy_table import Prices
from sqlalchemy import select

# Open a session directly
async def read_city(city:str):
    async with SessionLocal() as db:
        # Get a single price by city (Primary Key)
        city_price = await db.get(Prices, city)
        return f"Ticket price to {city} is ${city_price.price}" if city_price else "No price data available for this city"

In [4]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [7]:
await read_city("aaa")

'No price data available for this city'

In [8]:
price_function = {
    "name": "read_city",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'read_city',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['city'],
    'additionalProperties': False}}}]

In [ ]:
async def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "read_city":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('city')
            price_details = await read_city(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
async def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = await handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()